# 🌐 27 · Neural fields for heterogeneous structures: cryoDRGN
### A network that *is* the map — ask it for any 3D frequency and any conformation, and it answers

**BiologyToResearch** · Part F — Conformational heterogeneity: Gaussian mixtures and neural fields · Module 27 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/27_cryodrgn_neural_fields.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 3 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn, PyTorch |
| **Prerequisites** | [25 · Gaussian mixtures and reconstruction](25_gaussian_mixture_molecules.ipynb) (projection-slice theorem, poses); [26 · e2gmm](26_e2gmm_heterogeneity.ipynb) (latent spaces); [05 · Deep learning](05_deep_learning_pytorch.ipynb); [03 · Fourier transform](03_fourier_frequency.ipynb) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Zhong, Bepler, Berger & Davis 2021 (cryoDRGN, *Nat Methods*) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (2,000 particles of 32 × 32 pixels, networks
> with a few tens of thousands of weights, a minute of training). They illustrate the core ideas; they do **not**
> reproduce the paper's architecture or numbers, and Section 6 shows honestly what our tiny version cannot do. **All
> molecules and images are generated by code** (nothing is downloaded); our particles are cleaner than real ones (SNR 1)
> and have no CTF.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a **neural field** (coordinate network) is, and **show** with a one-dimensional example why it needs
   a **positional encoding** to represent fine detail.
2. **Describe** cryoDRGN's image model in Fourier space: the Hartley transform, the slice coordinates $R^{\top}\mathbf k$
   of every pixel, and a decoder $V(\mathbf k, \mathbf z)$ that returns the Fourier (Hartley) value of the map.
3. **Train** a homogeneous neural field from particles with known poses and **compare** it with direct Fourier
   reconstruction (Module 25).
4. **Explain** the variational autoencoder (VAE) of cryoDRGN — encoder, reparameterisation, KL divergence, ELBO, β — and
   **train** a miniature version on a mixture of assembly states.
5. **Analyse** a latent space the way cryoDRGN users do: PCA, k-means, maps generated at chosen latent points, and
   trajectories along a nearest-neighbour graph.
6. **Recognise** the method's needs and limits — consensus poses, data quantity, sparsely populated latent regions —
   and **compare** cryoDRGN's implicit representation with e2gmm's explicit Gaussians.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, PyTorch | 5 min |
| 1 · A network that is a function | neural fields; positional encoding in 1D | 15 min |
| 2 · Images as Fourier slices | Hartley transform, slice coordinates, a heterogeneous data set | 15 min |
| 3 · A homogeneous neural field | train $V(\mathbf k)$; compare with direct reconstruction; no-encoding control | 20 min |
| 4 · Adding a latent variable: the VAE | encoder, reparameterisation, KL, ELBO | 20 min |
| 5 · Reading the latent space | PCA, k-means, generated maps, a trajectory | 20 min |
| 6 · What the toy could not see | the swinging arm; why encoders need data | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Ribosomes are not built in one step: dozens of proteins and RNA helices join the growing particle in an order that
biochemists have tried to work out for fifty years. A cryo-EM sample of assembling bacterial large ribosomal subunits
(EMPIAR-10076, from Joey Davis's laboratory) contains a zoo of intermediates, some only partly folded. In 2021 Ellen Zhong,
Tristan Bepler, Bonnie Berger and Joseph Davis introduced **cryoDRGN**, which learns from such data a **continuous
generative model** of the structures: give it a point in a small latent space and it returns a 3D map. Their network
rediscovered the known assembly classes and found a new, rare one. The key ingredient is a kind of network that had just
transformed computer graphics: the **neural field**.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.sparse.csgraph import dijkstra
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import kneighbors_graph

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Helpers copied from [`course_toolkit.py`](../course_toolkit.py): the test protein of Part F, `move_domain`,
`gmm_project` / `gmm_to_map` (Module 25), `reconstruct_fourier`, rotations (Module 02) and `fsc` (Module 10).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)


def flexible_protein(seed: int = 7):
    """A code-generated ~80 kDa protein for heterogeneity experiments (Modules 25–30).

    Domains 0 and 1 form a rigid body; domain 2 (the 'arm') sits on domain 1 and can swing about a hinge; domain 3 is
    a small partner subunit that may be bound or not. Returns (atoms, hinge (x, y, z) in Å); atoms['domain'] labels
    every atom. Coordinates are in Å, centred near the origin.
    """
    rng = np.random.default_rng(seed)
    specs = [((21.0, 17.5, 16.0), (-19.0, -7.5, 0.0)), ((19.0, 16.0, 17.5), (19.0, -7.5, 0.0)),
             ((17.5, 12.5, 14.0), (21.0, 33.0, 0.0)), ((12.5, 11.0, 11.0), (-21.0, 26.0, 2.5))]
    parts = []
    for d, (radii, centre) in enumerate(specs):
        part = _transform(helix_bundle(radii, rng, chain="ABCD"[d]), _rz(rng.uniform(0, 2 * np.pi)), centre)
        part["domain"] = np.full(len(part["xyz"]), d)
        parts.append(part)
    atoms = merge_atoms(parts)
    shift = atoms["xyz"][atoms["domain"] < 3].mean(0)
    atoms["xyz"] = (atoms["xyz"] - shift).astype(np.float32)
    return atoms, np.array([19.0, 14.0, 0.0]) - shift


def move_domain(atoms, domain: int, R: np.ndarray, pivot) -> dict:
    """Copy of `atoms` with every atom of `domain` rotated by R (acting on (x, y, z)) about `pivot` (Å)."""
    out = dict(atoms)
    xyz = atoms["xyz"].astype(np.float64).copy()
    sel = atoms["domain"] == domain
    xyz[sel] = (xyz[sel] - pivot) @ np.asarray(R, float).T + pivot
    out["xyz"] = xyz.astype(np.float32)
    return out


def gmm_to_map(centres, amps, sigmas, box: int, voxel: float = 1.0) -> np.ndarray:
    """Density [z, y, x] (box³, origin at the box centre) of Σ_j A_j exp(−|x − c_j|² / 2σ_j²); c and σ in Å."""
    centres = np.asarray(centres, float)
    s = np.broadcast_to(np.asarray(sigmas, float), (len(centres),))
    g = (np.arange(box) - (box - 1) / 2.0) * voxel
    ex = [np.exp(-(g[None, :] - centres[:, a, None]) ** 2 / (2 * s[:, None] ** 2)) for a in range(3)]
    return np.einsum("n,nz,ny,nx->zyx", np.asarray(amps, float), ex[2], ex[1], ex[0], optimize=True).astype(np.float32)


def gmm_project(centres, amps, sigmas, R, box: int, pixel: float = 1.0) -> np.ndarray:
    """Projection [y, x] along z of a Gaussian mixture rotated by R (acting on (x, y, z)); same units as `gmm_to_map`.

    A 3D Gaussian projects to a 2D Gaussian of the same width, so the image is Σ_j w_j · g_y,j ⊗ g_x,j — two small
    matrices instead of a 3D grid (the 'project x and y separately' trick).
    """
    p = (np.asarray(centres, np.float32) @ np.asarray(R, np.float32).T)
    s = np.broadcast_to(np.asarray(sigmas, np.float32), (len(p),))[:, None]
    g = ((np.arange(box) - (box - 1) / 2.0) * pixel).astype(np.float32)[None, :]
    gx = np.exp(-(g - p[:, 0, None]) ** 2 / (2 * s ** 2))
    gy = np.exp(-(g - p[:, 1, None]) ** 2 / (2 * s ** 2))
    w = np.asarray(amps, np.float32)[:, None] * np.float32(np.sqrt(2 * np.pi)) * s / np.float32(pixel)   # ∫ dz
    return ((gy * w).T @ gx).astype(np.float32)


def reconstruct_fourier(images: np.ndarray, rotations, wiener: float = 1.0, chunk: int = 64) -> np.ndarray:
    """3D map [z, y, x] from projection images [n, y, x] with known rotations (R acts on (x, y, z), as in
    `rotate_volume`; image = rotate_volume(vol, R).sum(axis=0)).

    Projection-slice theorem: each image's 2D Fourier transform is the central slice of the 3D transform at
    k₃ = Rᵀ(kx, ky, 0). Slices are inserted with trilinear weights into a 2× padded grid, divided by the summed
    weights (+ `wiener`, in units of one image), transformed back and corrected for the interpolation kernel.
    """
    images = np.asarray(images, np.float64)
    n, N, _ = images.shape
    M, pad = 2 * N, N // 2
    k = np.arange(M) - M // 2
    ky, kx = np.meshgrid(k, k, indexing="ij")
    keep = (kx ** 2 + ky ** 2).ravel() <= (M // 2 - 1) ** 2
    plane = np.stack([kx.ravel(), ky.ravel(), np.zeros(M * M)], 1)[keep]
    shift_phase = np.exp(-1j * np.pi * (kx + ky) / M).ravel()[keep]          # half-pixel: image centre → index M/2
    Fr, Fi, W = np.zeros(M ** 3), np.zeros(M ** 3), np.zeros(M ** 3)
    corners = [(dz, dy, dx) for dz in (0, 1) for dy in (0, 1) for dx in (0, 1)]
    for start in range(0, n, chunk):
        idx_all, val_all, w_all = [], [], []
        for img, R in zip(images[start:start + chunk], list(rotations)[start:start + chunk]):
            big = np.zeros((M, M)); big[pad:pad + N, pad:pad + N] = img
            Fimg = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(big))).ravel()[keep] * shift_phase
            q = plane @ np.asarray(R, float)                                    # rows: (Rᵀ k)ᵀ in (x, y, z)
            pz = q[:, ::-1] + M // 2                                            # (z, y, x) grid coordinates
            i0 = np.floor(pz).astype(int); f = pz - i0
            for dz, dy, dx in corners:
                ii = i0 + (dz, dy, dx)
                wt = (np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1])
                      * np.where(dx, f[:, 2], 1 - f[:, 2]))
                ok = np.all((ii >= 0) & (ii < M), 1)
                idx_all.append((ii[ok, 0] * M + ii[ok, 1]) * M + ii[ok, 2]); val_all.append(wt[ok] * Fimg[ok])
                w_all.append(wt[ok])
        idx, val, wts = np.concatenate(idx_all), np.concatenate(val_all), np.concatenate(w_all)
        Fr += np.bincount(idx, val.real, M ** 3); Fi += np.bincount(idx, val.imag, M ** 3); W += np.bincount(idx, wts, M ** 3)
    F3 = ((Fr + 1j * Fi) / (W + wiener)).reshape(M, M, M)
    kz3, ky3, kx3 = np.meshgrid(k, k, k, indexing="ij")
    F3 *= np.exp(1j * np.pi * (kx3 + ky3 + kz3) / M)                            # centre back to (N − 1)/2
    vol = np.real(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(F3))))[pad:pad + N, pad:pad + N, pad:pad + N]
    x = (np.arange(N) - (N - 1) / 2.0) / M
    sinc2 = np.sinc(x) ** 2                                                     # trilinear-kernel correction
    return (vol / (sinc2[:, None, None] * sinc2[None, :, None] * sinc2[None, None, :])).astype(np.float32)
print("🧰 toolkit loaded")

---
## 1 · A network that is a function

> 📍 **Where we are.** Module 25 described a map by voxels or by Gaussians; Module 26 let a network move the Gaussians.
> cryoDRGN takes a third route: the map is the **network itself**. Module 28 carries this idea into cells (tomoDRGN).

A voxel map stores a number at every grid point. A **neural field** stores no grid at all: it is a small network
$V_\theta(\mathbf x)$ that takes a *coordinate* $\mathbf x$ and returns the value there. Training adjusts the weights
$\theta$ until the outputs match the data at the coordinates where we have data. Advantages: the "map" can be asked
about *any* coordinate (no fixed grid), it can be **conditioned** on extra inputs (a conformation!), and it is smooth
and differentiable.

There is a catch, best seen in 1D. Ordinary networks with ReLU units learn smooth, slowly varying functions first and
struggle with rapid oscillations (**spectral bias**). The fix used by neural fields — and by cryoDRGN — is a
**positional encoding**: instead of $x$, feed the network many sines and cosines of $x$ at increasing frequencies,

$$\gamma(x) = \big(\sin(\omega_1 x), \cos(\omega_1 x), \dots, \sin(\omega_F x), \cos(\omega_F x)\big),\qquad
  \omega_f = 2\pi\,(D/2)^{(f-1)/(F-1)},$$

*in words:* a ladder of wavelengths from the whole box down to two pixels (the "geometric" encoding of the cryoDRGN
paper; the current cryoDRGN code uses random Gaussian frequencies by default — same idea). Let us fit a wiggly 1D
signal both ways.

In [ ]:
# ── 1.1 Spectral bias in 1D: raw coordinate vs positional encoding ──
D1 = 64
x1 = torch.linspace(-0.5, 0.5, 256)[:, None]
target1 = torch.sin(2 * math.pi * 3 * x1) + 0.5 * torch.sin(2 * math.pi * 17 * x1 + 1.0)   # slow + fast wave
W1 = torch.tensor(2 * math.pi * (D1 / 2) ** (np.arange(8) / 7), dtype=torch.float32)


def encode_1d(x):
    """Geometric positional encoding of 1D coordinates (n, 1) → (n, 16)."""
    return torch.cat([torch.sin(x * W1), torch.cos(x * W1)], 1)


fits = {}
for name, feat in [("raw coordinate", lambda x: x), ("positional encoding", encode_1d)]:
    torch.manual_seed(SEED)
    net = nn.Sequential(nn.Linear(feat(x1).shape[1], 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for it in range(1500):
        loss = ((net(feat(x1)) - target1) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    fits[name] = (net(feat(x1)).detach(), loss.item())
    print(f"{name:20s}: final squared error {loss.item():.4f}")

fig, ax = plt.subplots(figsize=(7.5, 2.8))
ax.plot(x1, target1, color="k", lw=2, label="target")
for (name, (y, l)), col in zip(fits.items(), ["tab:red", "tab:green"]):
    ax.plot(x1, y, color=col, label=f"{name} (error {l:.3f})")
ax.set_xlabel("coordinate x"); ax.legend(fontsize=8); ax.set_title("Same network, same training — different inputs")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the raw coordinate the network learns the slow wave and smooths over the fast one; with
the positional encoding it captures both. A 3D map in Fourier space is full of oscillations (every atom contributes a
wave), so cryoDRGN always encodes its coordinates this way.

---
## 2 · Images as Fourier slices

### 2.1 The Hartley transform

cryoDRGN works in Fourier space, where the projection-slice theorem (Module 25) turns "project the 3D map at pose $R$"
into "read the 3D transform on the plane $R^{\top}(k_x, k_y, 0)$". A network outputs real numbers, but Fourier
coefficients are complex. The authors use the **Hartley transform**,

$$H(\mathbf k) = \operatorname{Re}F(\mathbf k) - \operatorname{Im}F(\mathbf k),$$

*in words:* one real number per frequency that holds exactly the same information as the complex coefficient (because
for a real image $F(-\mathbf k)$ is the complex conjugate of $F(\mathbf k)$, the pair $H(\mathbf k), H(-\mathbf k)$
gives back both the real and the imaginary part). The projection-slice theorem holds for Hartley transforms too.

⚠️ A practical detail: the FFT puts the origin at pixel $D/2$, our images are centred at $(D-1)/2$. A half-pixel shift
is a phase twist that would differ from pose to pose; we remove it (`ht2` below), exactly as `reconstruct_fourier` does.

In [ ]:
# ── 2.1 Hartley transform of 2D images and of 3D volumes ──
N, PX = 32, 5.0                                                   # 32 × 32 pixels of 5 Å: a 160 Å box
KC = np.arange(N) - N // 2                                        # centred integer frequencies


def ht2(imgs: np.ndarray) -> np.ndarray:
    """Centred 2D Hartley transform of images (..., N, N), with the half-pixel centring correction."""
    F = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(imgs, axes=(-2, -1))), axes=(-2, -1))
    F = F * np.exp(-1j * np.pi * (KC[:, None] + KC[None, :]) / N)
    return (F.real - F.imag).astype(np.float32)


def iht3(H: np.ndarray) -> np.ndarray:
    """Volume [z, y, x] from its centred 3D Hartley transform (inverse of the 3D version of ht2)."""
    Hm = np.roll(np.flip(H, axis=(0, 1, 2)), 1, axis=(0, 1, 2))   # H(−k) on the centred grid
    F = (H + Hm) / 2 - 1j * (H - Hm) / 2                           # back to complex Fourier coefficients
    kz, ky, kx = np.meshgrid(KC, KC, KC, indexing="ij")
    F = F * np.exp(1j * np.pi * (kx + ky + kz) / N)
    return np.real(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(F)))).astype(np.float32)


# self-test: transform a (band-limited) volume to Hartley space and back
test_vol = lowpass(rng.normal(size=(N, N, N)).astype(np.float32), 0.4)
F3 = np.fft.fftshift(np.fft.fftn(np.fft.ifftshift(test_vol)))
kz_, ky_, kx_ = np.meshgrid(KC, KC, KC, indexing="ij")
F3 = F3 * np.exp(-1j * np.pi * (kx_ + ky_ + kz_) / N)
print(f"Hartley round trip: max error {np.abs(iht3(F3.real - F3.imag) - test_vol).max() / np.abs(test_vol).max():.1e} "
      f"(relative to the largest value)")

### 2.2 A heterogeneous data set: three assembly states and a swinging arm

In the spirit of the assembly data, our protein comes in three **assembly states** — the **core** (the two body
domains), **core + arm**, and the **complete** protein with its partner — in equal numbers. Whenever the arm is present it
also swings by a random angle between −30° and +30° (as in Module 26). 2,000 particles, random known poses, SNR 1.

In [ ]:
# ── 2.2 Simulate 2,000 particles: 3 assembly states × a swinging arm ──
atoms, hinge = flexible_protein()
Z_ATOM = np.array([{"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0}.get(e, 6.0) for e in atoms["element"]], np.float32)
SIG_ATOM, N_PART, SNR = 2.5, 2000, 1.0
STATE_NAMES = ["core", "core + arm", "complete"]


def conformation(theta_deg: float, state: int):
    """Atom coordinates and amplitudes for an assembly state (0, 1, 2) with the arm turned by theta."""
    moved = move_domain(atoms, 2, rot_z(np.radians(theta_deg)), hinge)
    d = atoms["domain"]
    present = (d <= 1) | ((d == 2) & (state >= 1)) | ((d == 3) & (state >= 2))
    return moved["xyz"], (Z_ATOM * present).astype(np.float32)


state_true = rng.integers(0, 3, N_PART)
theta_true = rng.uniform(-30, 30, N_PART)
rot_true = np.array([random_rotation(rng) for _ in range(N_PART)]).astype(np.float32)
t0 = time.time()
clean = np.array([gmm_project(*conformation(theta_true[i], state_true[i]), SIG_ATOM, rot_true[i], N, PX)
                  for i in range(N_PART)])
particles = (clean + rng.normal(0, np.sqrt(clean.var() / SNR), clean.shape)).astype(np.float32)
print(f"{N_PART} particles in {time.time() - t0:.1f} s; states {np.bincount(state_true)}; SNR {SNR}")

fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for j in range(6):
    axes[j].imshow(particles[j]); axes[j].set_title(f"{STATE_NAMES[state_true[j]]}", fontsize=8); axes[j].axis("off")
plt.tight_layout(); plt.show()

### 2.3 Where each pixel lives in 3D Fourier space

For particle $i$, pixel frequency $\mathbf k = (k_x, k_y, 0)$ sits at $R_i^{\top}\mathbf k$ in the molecule's Fourier
space. We keep the frequencies inside a circle of radius 12 (out of 16) — resolution $160/12 \approx 13$ Å — and express
them in "cycles per box", between −0.5 and 0.5, as cryoDRGN does.

In [ ]:
# ── 2.3 Hartley targets and slice coordinates ──
RMASK = 12
KY2, KX2 = np.meshgrid(KC, KC, indexing="ij")
in_circle = (KX2 ** 2 + KY2 ** 2) <= RMASK ** 2
H_all = ht2(particles)[:, in_circle]                              # (N_PART, M) Hartley values inside the circle
H_SCALE = H_all.std()
H_t = torch.tensor(H_all / H_SCALE)                               # normalised, as cryoDRGN normalises its data
LATTICE = torch.tensor(np.stack([KX2[in_circle], KY2[in_circle], np.zeros(in_circle.sum())], 1) / N,
                       dtype=torch.float32)                       # (M, 3): (kx, ky, 0) in cycles per box
R_t = torch.tensor(rot_true)


def slice_coords(R):
    """3D coordinates Rᵀk of every lattice point for a batch of poses R (b, 3, 3) → (b, M, 3)."""
    return torch.einsum("mj,bji->bmi", LATTICE, R)


print(f"{in_circle.sum()} Fourier coefficients per image (out of {N * N}); Hartley values scaled by {H_SCALE:.1f}")

---
## 3 · A homogeneous neural field

First, no heterogeneity at all: one network $V(\mathbf k)$ for the average molecule. For a batch of particles we compute
the slice coordinates, encode them (Section 1, now in 3D: 8 frequencies × 3 axes × sin/cos = 48 inputs), run the network
and compare its outputs with the particles' Hartley values by **mean squared error**. When training is done we evaluate
the network on a full 3D grid of frequencies and inverse-transform: a 3D map.

In [ ]:
# ── 3.1 The decoder V(k, z): positional encoding + a small MLP ──
W3 = torch.tensor(2 * math.pi * (N / 2) ** (np.arange(8) / 7), dtype=torch.float32)


def encode_3d(k):
    """Geometric positional encoding of 3D frequency coordinates (..., 3) → (..., 48)."""
    a = (k[..., :, None] * W3).reshape(*k.shape[:-1], -1)
    return torch.cat([torch.sin(a), torch.cos(a)], -1)


class FieldDecoder(nn.Module):
    """V(k, z): Hartley value of the map at 3D frequency k, for latent point z (z_dim may be 0)."""

    def __init__(self, z_dim: int = 0, hidden: int = 128, encode: bool = True):
        super().__init__()
        self.encode, self.z_dim = encode, z_dim
        n_in = (48 if encode else 3) + z_dim
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, k, z=None):
        f = encode_3d(k) if self.encode else k
        if self.z_dim:
            f = torch.cat([f, z[:, None, :].expand(-1, k.shape[1], -1)], -1)
        return self.net(f)[..., 0]


GRID3 = np.stack(np.meshgrid(KC, KC, KC, indexing="ij"), -1)          # (z, y, x) integer frequencies
IN_BALL = (GRID3 ** 2).sum(-1) <= RMASK ** 2
BALL_T = torch.tensor(GRID3[IN_BALL][:, ::-1] / N, dtype=torch.float32)   # (kx, ky, kz) in cycles per box


def field_to_map(decoder, z=None) -> np.ndarray:
    """Evaluate the decoder on the 3D frequency ball and inverse-Hartley-transform it into a map [z, y, x]."""
    with torch.no_grad():
        zz = None if z is None else torch.as_tensor(np.asarray(z), dtype=torch.float32)[None]
        vals = decoder(BALL_T[None], zz)[0].numpy() * H_SCALE
    H = np.zeros((N, N, N), np.float32)
    H[IN_BALL] = vals
    return iht3(H)


print("decoder parameters:", sum(p.numel() for p in FieldDecoder().parameters()))

In [ ]:
# ── 3.2 Train homogeneous fields with and without positional encoding ──
def train_homogeneous(encode: bool, epochs: int = 6):
    """Fit V(k) to all particles (MSE in Hartley space). Returns the decoder."""
    torch.manual_seed(SEED)
    dec = FieldDecoder(0, encode=encode)
    opt = torch.optim.Adam(dec.parameters(), lr=1e-3)
    t0 = time.time()
    for ep in range(epochs):
        for idx in torch.randperm(N_PART).split(16):
            loss = ((dec(slice_coords(R_t[idx])) - H_t[idx]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    print(f"positional encoding {'on ' if encode else 'off'}: {epochs} epochs in {time.time() - t0:.0f} s, "
          f"final MSE {loss.item():.3f}")
    return dec


average_truth = np.mean([gmm_to_map(*conformation(0.0, s), SIG_ATOM, N, PX) for s in range(3)], axis=0)
maps3 = {"direct (Module 25)": reconstruct_fourier(particles, rot_true),
         "neural field": field_to_map(train_homogeneous(True)),
         "neural field, no encoding": field_to_map(train_homogeneous(False))}
fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.8))
axes[0].imshow(lowpass(average_truth, PX * RMASK / (N * PX)).sum(0)); axes[0].set_title("average truth (13 Å)", fontsize=9)
for ax, (name, m) in zip(axes[1:], maps3.items()):
    f, c, _ = fsc(m, average_truth, PX)
    ax.imshow(m.sum(0)); ax.set_title(f"{name}\nFSC(0.5) {resolution_at(f, c, 0.5):.0f} Å", fontsize=9)
    print(f"{name:26s}: correlation with the average truth {pearson(m, average_truth):.3f}, "
          f"FSC(0.5) {resolution_at(f, c, 0.5):.1f} Å")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In a few seconds of training, the network learned a 3D map from 2D pictures — never seeing a
voxel. It is a little behind direct reconstruction at this tiny scale (a few epochs; the paper trains for tens of
epochs on GPUs), but the point is made: **the map can be a function**. Without positional encoding the same network
gets stuck at low resolution (spectral bias, Section 1). (Each map is an *average*: our particles are a mixture of three
assembly states.)

---
## 4 · Adding a latent variable: the variational autoencoder

Now let the map depend on a **latent point** $\mathbf z$ (a few numbers): $V(\mathbf k, \mathbf z)$. Different particles
get different $\mathbf z$, hence different maps. Where does $\mathbf z_i$ come from? From an **encoder** network that
reads the particle image. cryoDRGN trains encoder and decoder together as a **variational autoencoder (VAE)**:

1. The encoder outputs, for image $X_i$, not one point but a small Gaussian cloud: a mean $\boldsymbol\mu_i$ and a
   variance $\boldsymbol\sigma_i^2$ — "the conformation is probably here, give or take".
2. **Reparameterisation:** draw $\mathbf z_i = \boldsymbol\mu_i + \boldsymbol\sigma_i\odot\boldsymbol\epsilon$ with
   $\boldsymbol\epsilon\sim\mathcal N(0, I)$ — random, yet differentiable with respect to $\boldsymbol\mu_i$ and
   $\boldsymbol\sigma_i$.
3. The decoder renders the slice at pose $R_i$ from $\mathbf z_i$; the **reconstruction error** (MSE) is measured.
4. A **KL divergence** term pulls each cloud toward the standard Gaussian $\mathcal N(0, I)$:
   $\mathrm{KL}=\tfrac12\sum_d(\mu_d^2+\sigma_d^2-\log\sigma_d^2-1)$.

$$\mathcal L \;=\; \underbrace{\mathrm{MSE}\big(V(R_i^{\top}\mathbf k, \mathbf z_i),\,H_i(\mathbf k)\big)}_{\text{fit the image}}
  \;+\; \beta\,\frac{\mathrm{KL}\big(q(\mathbf z\mid X_i)\,\Vert\,\mathcal N(0, I)\big)}{M},$$

*in words:* explain each image as well as possible, but keep the latent clouds tidy and overlapping, so that the latent
space is smooth and every region of it decodes to a plausible map. (Minimising $\mathcal L$ maximises the **evidence
lower bound**, ELBO, of variational inference.) In the cryoDRGN code the KL term is divided by the number of pixels $M$
and weighted by $\beta = 1/\text{(latent dimension)}$ by default; we do the same.

🧮 One difference from the paper: its encoder is a fully connected network on the image; ours is a small
**convolutional** network (Module 05), which learns faster from our few thousand images. The poses $R_i$ are given —
cryoDRGN takes them from a consensus refinement and does not re-estimate them.

In [ ]:
# ── 4.1 The encoder and the VAE training loop ──
Z_DIM = 2
X_enc = torch.tensor(particles / particles.std())[:, None]        # encoder input: the normalised image


class ImageEncoder(nn.Module):
    """Small CNN: image (1, N, N) → mean and log-variance of q(z | image)."""

    def __init__(self, z_dim: int):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                 nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 2 * z_dim))

    def forward(self, x):
        out = self.net(x)
        return out[:, :self.z_dim], out[:, self.z_dim:]


torch.manual_seed(SEED)
encoder, decoder = ImageEncoder(Z_DIM), FieldDecoder(Z_DIM)
opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
BETA, M_PIX = 1.0 / Z_DIM, H_t.shape[1]
log, t0 = [], time.time()
for ep in range(15):
    for idx in torch.randperm(N_PART).split(16):
        mu, logvar = encoder(X_enc[idx])
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)                     # reparameterisation
        mse = ((decoder(slice_coords(R_t[idx]), z) - H_t[idx]) ** 2).mean()
        kl = (0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(1)).mean()
        loss = mse + BETA * kl / M_PIX
        opt.zero_grad(); loss.backward(); opt.step()
    log.append((mse.item(), kl.item()))
    if ep % 5 == 4:
        print(f"epoch {ep + 1:2d}: MSE {mse.item():.3f}, KL {kl.item():.2f} nats  [{time.time() - t0:.0f} s]")
with torch.no_grad():
    z_mean = encoder(X_enc)[0].numpy()                             # cryoDRGN analyses the encoder means

The KL term grows from zero as the encoder starts to put information into the latent space; the MSE falls as the
decoder uses it. (The MSE cannot reach zero: half of each image's variance is noise at SNR 1.)

---
## 5 · Reading the latent space

A trained cryoDRGN model is explored, not just scored. The standard tools (all in the cryoDRGN software): plot the
latent means (after PCA or UMAP if the latent has more than two dimensions), cluster them (k-means), **generate maps**
at chosen latent points, and follow **trajectories** through the latent space.

In [ ]:
# ── 5.1 Latent means, coloured by the hidden assembly state; k-means with 3 clusters ──
km = KMeans(3, n_init=10, random_state=SEED).fit(z_mean)
C = np.zeros((3, 3), int); np.add.at(C, (km.labels_, state_true), 1)
row, col = linear_sum_assignment(-C)
acc_states = C[row, col].sum() / N_PART
print("contingency (rows = k-means clusters, columns = hidden states):\n", C)
print(f"best-matched accuracy of 3 k-means clusters vs hidden assembly states: {acc_states:.2f} (chance 0.33)")
fig, ax = plt.subplots(figsize=(5, 3.8))
for s, colr in zip(range(3), ["tab:blue", "tab:orange", "tab:green"]):
    sel = state_true == s
    ax.scatter(z_mean[sel, 0], z_mean[sel, 1], s=5, color=colr, alpha=0.6, label=STATE_NAMES[s])
ax.scatter(*km.cluster_centers_.T, marker="x", s=120, color="k", label="k-means centres")
ax.set_xlabel("z₁"); ax.set_ylabel("z₂"); ax.legend(fontsize=7)
ax.set_title("latent means (colour = hidden state)", fontsize=9); plt.tight_layout(); plt.show()
assert acc_states > 0.6, "the latent space should separate the assembly states"

In [ ]:
# ── 5.2 Generate a map at each k-means centre ──
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
gen_maps = [field_to_map(decoder, c) for c in km.cluster_centers_]
mass = np.array([m[m > 0].sum() for m in gen_maps])
for ax, k in zip(axes, np.argsort(mass)):
    ax.imshow(gen_maps[k].sum(0)); ax.axis("off")
    ax.set_title(f"cluster {k}: mostly '{STATE_NAMES[np.argmax(C[k])]}'", fontsize=9)
plt.suptitle("Maps generated by the decoder at the three cluster centres (projections down z)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The encoder was never told about assembly states, yet the latent space falls into groups
that match them, and the decoder, asked for a map at each group's centre, draws the core, the core with the arm, and
the complete protein. This is the logic of cryoDRGN's assembly analysis: the authors report that their latent space
recovered the known classes of assembling large ribosomal subunits and revealed a new, rare one.

### 5.3 A trajectory through the latent space

To watch one state turn into another, cryoDRGN can connect latent points through the data: build a **k-nearest-neighbour
graph** of the particles' latent means, find the **shortest path** between two chosen particles (Dijkstra's algorithm),
and generate maps along it. Following the data, rather than a straight line, keeps the path in populated regions.

In [ ]:
# ── 5.3 Shortest path on a kNN graph from the 'core' cluster to the 'complete' cluster ──
order = np.argsort(mass)                                                   # clusters sorted by generated mass
start = int(np.argmin(np.linalg.norm(z_mean - km.cluster_centers_[order[0]], axis=1)))
end = int(np.argmin(np.linalg.norm(z_mean - km.cluster_centers_[order[2]], axis=1)))
graph = kneighbors_graph(z_mean, 10, mode="distance")
_, pred = dijkstra(graph, directed=False, indices=start, return_predecessors=True)
path = [end]
while path[-1] != start:
    path.append(int(pred[path[-1]]))
path = path[::-1]
picks = [path[int(round(t))] for t in np.linspace(0, len(path) - 1, 5)]
print(f"path of {len(path)} particles; states along it (hidden): {[STATE_NAMES[state_true[p]] for p in picks]}")
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4), gridspec_kw=dict(width_ratios=[1.4, 1, 1, 1, 1, 1]))
axes[0].scatter(z_mean[:, 0], z_mean[:, 1], s=2, color="0.75"); axes[0].plot(*z_mean[path].T, "r-", lw=1.5)
axes[0].scatter(*z_mean[picks].T, color="k", s=20, zorder=3); axes[0].set_title("path in latent space", fontsize=8)
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, p in zip(axes[1:], picks):
    ax.imshow(field_to_map(decoder, z_mean[p]).sum(0)); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 Along the path the generated maps change gradually from the core alone toward the complete protein — a movie
assembled from particles that actually exist along the way. ⚠️ The movie is a *path through data*, not a measured
time course: assembly order is suggested, not proven.

---
## 6 · What the toy could not see

Our particles also had a **swinging arm**. Did the latent space notice it? Within each state that has an arm, we
correlate the latent coordinates with the hidden swing angle.

In [ ]:
# ── 6.1 Is the arm angle in the latent space? ──
for s in [1, 2]:
    sel = state_true == s
    rhos = [spearmanr(z_mean[sel, d], theta_true[sel]).correlation for d in range(Z_DIM)]
    print(f"state '{STATE_NAMES[s]}': Spearman correlation of z₁, z₂ with the hidden arm angle: "
          f"{rhos[0]:+.2f}, {rhos[1]:+.2f}")

🔍 **What just happened?** Essentially nothing: our miniature cryoDRGN sorted the *composition* but did not organise the
*swing*. Why, when Module 26's e2gmm-lite found the same swing in fewer particles?

- **The encoder must see past orientation.** Losing or gaining a domain changes how much density an image contains,
  whatever the viewing direction — easy to detect. A 10–15 Å shift of the arm looks completely different from every
  direction; to read it from an image alone, the encoder must effectively learn the whole range of orientations. That
  takes many examples: cryoDRGN papers use 10⁴–10⁵ particles per data set (we have 2,000), and on simulated data with
  enough particles the authors report a latent coordinate that tracks a linear motion almost perfectly (Spearman
  correlation of magnitude 0.996).
- **e2gmm's fingerprints already "know" the pose.** Module 26's encoder read FRC gradients computed at each particle's
  pose, in the molecule's frame — orientation was removed before the network ever saw the data.
- **More views of the same particle help.** In a tomogram every particle is imaged at many tilts. Module 28 shows how
  tomoDRGN pools them.

✏️ **Your turn.** Make the job easier — raise `SNR` to `3.0`, or `N_PART` to `4000` (runtime roughly doubles), or widen
the swing to ±60° in 2.2 — and re-run Sections 2–6. Does the swing start to show up in the latent space? Which change
helps most?

### 📄 Paper connection — Zhong *et al.* 2021: cryoDRGN

**Paper:** Zhong ED, Bepler T, Berger B, Davis JH. *CryoDRGN: reconstruction of heterogeneous cryo-EM structures using
neural networks.* Nature Methods 18:176–185 (2021). [doi:10.1038/s41592-020-01049-4](https://doi.org/10.1038/s41592-020-01049-4)
(preprint [bioRxiv 10.1101/2020.03.27.003871](https://doi.org/10.1101/2020.03.27.003871))

- **Question it answers:** how to reconstruct a *continuous distribution* of 3D structures — discrete states and
  continuous motions alike — from a single cryo-EM data set, without choosing the number of classes in advance.
- **What the paper actually does:** image model = projection at pose $R$, CTF, in-plane shift and Gaussian noise; by the
  Fourier slice theorem the image's D × D lattice, rotated by $R$, gives the 3D coordinates of its slice. The decoder is an
  MLP $V(\mathbf k, \mathbf z)$ on a positional encoding of the 3D Fourier coordinate (sines and cosines at geometric
  wavelengths in the paper; the current code defaults to random Gaussian Fourier features) concatenated with $\mathbf z$,
  returning Hartley-space values. The encoder is an MLP on the image returning the mean and (diagonal) variance of
  $q(\mathbf z\mid X)$; the prior is $\mathcal N(0, I)$; training maximises the ELBO (code: MSE + β·KL / number of pixels,
  β = 1/zdim by default) with poses from a consensus refinement (cryoSPARC), Adam, batch size 8, 25–150 epochs, images
  downsampled to 96–360 pixels, latent dimension 1 or 10.
- **Analysis tools:** latent means; PCA/UMAP; k-means (k = 20) to generate representative maps; trajectories via a
  nearest-neighbour graph and Dijkstra's shortest path.
- **Result reported:** on simulated data, a 1D latent tracking a linear motion (Spearman correlation of magnitude
  0.996) and 99.9 % separation of three discrete complexes (30S/50S/70S); on real data, a rotated 40S head in the
  *Plasmodium falciparum* 80S ribosome (EMPIAR-10028), the assembly classes of the bacterial large ribosomal subunit plus
  a new class (≈ 1,200 particles; EMPIAR-10076), and a continuous motion of the SF3b/helicase region along the first
  principal component in a spliceosome (EMPIAR-10180).
- **What our toy keeps:** Fourier-slice coordinates, Hartley targets, positional encoding, coordinate-MLP decoder,
  VAE with reparameterisation and β-weighted KL, latent analysis with k-means, generated maps and kNN/Dijkstra paths.
- **What our toy leaves out:** CTF, translations, a fully connected encoder (ours is convolutional), UMAP, GPU-scale data,
  and the paper's training lengths — which is why our latent misses the swing (Section 6).
- **Limitations stated:** depends on consensus poses; sparsely populated latent regions are not guaranteed to be
  physical; junk particles must be filtered; hyper-parameters matter.
- **Code:** <https://github.com/zhonge/cryodrgn> (`cryodrgn train_vae`, `cryodrgn analyze`).

### 💡 Two ways to describe a moving molecule

| | e2gmm (Module 26) | cryoDRGN (this module) |
|---|---|---|
| representation | explicit: Gaussians you can point at | implicit: a network you query |
| what the decoder outputs | positions/amplitudes of Gaussians | Fourier (Hartley) values at any 3D frequency |
| encoder input | FRC gradients at the known pose (pose-free) | the image itself |
| strengths | interpretable motions, rigid bodies, link to atomic models (Module 30) | no assumption about shape; any kind of change |
| care needed | Gaussians must be able to represent the change | latent regions without particles; data hunger |

---
## ✅ Checkpoint

1. What does the cryoDRGN decoder take as input, and what does it output?
   <details><summary>Answer</summary>A 3D Fourier coordinate (positionally encoded) and a latent point z; it outputs the Hartley-space value of the map with conformation z at that frequency.</details>
2. Why does a neural field need a positional encoding?
   <details><summary>Answer</summary>Plain ReLU networks learn smooth functions first (spectral bias); feeding sines and cosines of the coordinates at many frequencies lets them represent rapid oscillations — fine detail.</details>
3. For particle i with pose R_i, where are its pixels in 3D Fourier space?
   <details><summary>Answer</summary>At R_iᵀ(kx, ky, 0) for every pixel frequency (kx, ky): a central plane tilted by the pose.</details>
4. What do the two terms of the VAE loss do?
   <details><summary>Answer</summary>The reconstruction term makes the decoded slice match the image; the KL term pulls each particle's latent distribution toward N(0, I), keeping the latent space compact and smooth.</details>
5. Why follow a kNN-graph shortest path rather than a straight line between two latent points?
   <details><summary>Answer</summary>The path stays where particles actually are; a straight line may cross empty regions, where the decoder's output is unconstrained by data.</details>
6. Why did our toy find the assembly states but not the arm swing?
   <details><summary>Answer</summary>Composition changes are visible from any direction; a small motion must be recognised through orientation, which an image encoder learns only from many more particles.</details>

## 🧠 Questions & discussion

1. cryoDRGN takes poses from a consensus refinement of *all* particles. When could those poses be systematically wrong
   for one state, and what would the latent space then show?
2. A generated map at a latent point with very few particles nearby looks sharp and convincing. Should you believe it?
3. The rare new assembly class reported in the paper had about 1,200 particles out of more than 100,000. What makes a
   rare state detectable, and how would you validate it?
4. Compare an implicit neural field with a voxel map: which is easier to store, to interpolate between conformations, to
   validate with half-maps?
5. Would a latent dimension of 1 or of 10 have been better for our three states? What do you trade?

## 🚀 Bonus — going further

- **Random Fourier features.** Replace the geometric encoding by $\gamma(\mathbf k)=[\sin(2\pi B\mathbf k), \cos(2\pi B\mathbf k)]$
  with a random Gaussian matrix $B$ (the current cryoDRGN default) and compare the homogeneous maps.
- **A fully connected encoder.** Swap `ImageEncoder` for an MLP on the Hartley values (as in the paper) and compare the
  state accuracy in 5.1.
- **Junk particles.** Add 200 pure-noise "particles" with random poses. Where do they land in the latent space?
- **Real software:** the cryoDRGN tutorial (<https://github.com/zhonge/cryodrgn>) walks through EMPIAR-10076.

## 📝 Lab note

```text
Module 27 — date:
Input: 2,000 simulated particles (3 assembly states, arm swing ±30°), 32 × 32 px at 5 Å, SNR 1, known poses
Output: homogeneous neural fields (with/without encoding), VAE latent space, generated maps, kNN/Dijkstra path
Method: Hartley transform, Fourier-slice coordinates, positional encoding, coordinate MLP, VAE (β·KL/M), k-means
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Zhong, Bepler, Berger & Davis 2021 (cryoDRGN)
```

## 📚 References

1. Zhong ED, Bepler T, Berger B, Davis JH. CryoDRGN: reconstruction of heterogeneous cryo-EM structures using neural networks. *Nat Methods* 18:176–185 (2021). doi:10.1038/s41592-020-01049-4
2. Kingma DP, Welling M. Auto-encoding variational Bayes. *ICLR* (2014). arXiv:1312.6114
3. Tancik M *et al.* Fourier features let networks learn high frequency functions in low dimensional domains. *NeurIPS* (2020). arXiv:2006.10739
4. Mildenhall B *et al.* NeRF: representing scenes as neural radiance fields for view synthesis. *ECCV* (2020). arXiv:2003.08934
5. Bracewell RN. The fast Hartley transform. *Proc IEEE* 72:1010–1018 (1984). doi:10.1109/PROC.1984.12968
6. Davis JH *et al.* Modular assembly of the bacterial large ribosomal subunit. *Cell* 167:1610–1622 (2016). doi:10.1016/j.cell.2016.11.020

## ➡️ Next

**[28 · Heterogeneity inside cells: tomoDRGN and sub-tilt series](28_tomodrgn_subtilts.ipynb)** — the same neural field,
fed with all the tilted views of each particle in a tomogram.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")